## Exploring Wilcox 2022 test suites
The goal of this notebook is to identify the POS tags for starting words of filled-gap regions and post-gap regions in some of the test suite data used in Wilcox et al 2022 (https://github.com/wilcoxeg/fgd_learnability/tree/master)

In [1]:
import spacy
import pandas as pd
from collections import Counter

nlp = spacy.load("en_core_web_sm")


/home/marrsia/.local/lib/python3.8/site-packages/thinc/compat.py:36: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  hasattr(torch, "has_mps")
/home/marrsia/.local/lib/python3.8/site-packages/thinc/compat.py:37: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  and torch.has_mps  # type: ignore[attr-defined]


In [2]:
def get_first_word_pos_in_context(full_sentence, region):
    """
    Gets POS of first word of region using full sentence as context.
    """
    first_word = region.split()[0]
    doc = nlp(full_sentence)
    
    for token in doc:
        if token.text.lower() == first_word.lower():
            return token.pos_
    
    return None

In [3]:
def analyse_region_first_word_pos(csv_path, region_cols):
    """
    For each row, reconstructs the full sentence for context using all columns
    except 'item' and 'condition', and gets POS of the first word of each region.

    Args:
        csv_path: path to CSV file
        region_cols: list of column names to analyse e.g. ["np2", "prep"]

    Returns:
        dict of {col_name: Counter of POS tags}
    """
    df = pd.read_csv(csv_path)
    sentence_cols = [col for col in df.columns if col not in ["item", "condition"]]
    
    pos_counters = {col: Counter() for col in region_cols}
    
    for _, row in df.iterrows():
        full_sentence = " ".join(
            str(row[col]).strip() for col in sentence_cols
            if pd.notna(row[col]) and str(row[col]).strip()
        )
        doc = nlp(full_sentence)
        token_pos = {token.text.lower(): token.pos_ for token in doc}
        
        for col in region_cols:
            value = str(row[col]).strip() if pd.notna(row[col]) else ""
            if value:
                first_word = value.split()[0].lower()
                pos = token_pos.get(first_word)
                if pos:
                    pos_counters[col][pos] += 1
    
    return pos_counters

In [4]:
WILCOX_FILE_CONFIGS = {
    "basic_object.csv":  {"gap": ["np2"],                           "post_gap": ["prep"]},
    "basic_pp.csv":      {"gap": ["np3"],                           "post_gap": ["end"]},
    "basic_subject.csv": {"gap": ["np1"],                           "post_gap": ["verb"]},
    "embed1.csv":        {"gap": ["gap"],                           "post_gap": ["continuation"]},
    "embed2.csv":        {"gap": ["gap"],                           "post_gap": ["continuation"]},
    "embed3.csv":        {"gap": ["gap"],                           "post_gap": ["continuation"]},
    "embed4.csv":        {"gap": ["gap"],                           "post_gap": ["continuation"]},
    "hierarchy.csv":     {"gap": ["subject_gap", "matrix_gap"],     "post_gap": ["filler", "continuation"]},
}

In [5]:
import os

wilcox_dir = "../data/wilcox2022_test_suites"

all_results = {}

for filename, config in WILCOX_FILE_CONFIGS.items():
    filepath = os.path.join(wilcox_dir, filename)
    if not os.path.exists(filepath):
        print(f"Warning: {filename} not found, skipping")
        continue
    print(f"Processing {filename}...")
    all_results[filename] = {
        "gap":      analyse_region_first_word_pos(filepath, config["gap"]),
        "post_gap": analyse_region_first_word_pos(filepath, config["post_gap"])
    }

# consolidate gap and post_gap separately
consolidated_gap = Counter()
consolidated_post_gap = Counter()

for filename, result in all_results.items():
    for col, counter in result["gap"].items():
        consolidated_gap += counter
    for col, counter in result["post_gap"].items():
        consolidated_post_gap += counter

print("\nGap region first word POS:")
for pos, count in consolidated_gap.most_common():
    print(f"  {pos}: {count}")

print("\nPost-gap region first word POS:")
for pos, count in consolidated_post_gap.most_common():
    print(f"  {pos}: {count}")

Processing basic_object.csv...
Processing basic_pp.csv...
Processing basic_subject.csv...
Processing embed1.csv...
Processing embed2.csv...
Processing embed3.csv...
Processing embed4.csv...
Processing hierarchy.csv...

Gap region first word POS:
  DET: 922
  PRON: 322
  NOUN: 14
  ADJ: 10
  VERB: 4

Post-gap region first word POS:
  ADP: 1370
  ADJ: 390
  VERB: 206
  NOUN: 104
  ADV: 64
  SCONJ: 22
  NUM: 20
  PART: 4
  DET: 4


In [8]:
gap_df = counter_to_df(consolidated_gap, 'no_gap')[['count']].rename(columns={'count': 'no_gap'})
post_gap_df = counter_to_df(consolidated_post_gap, 'gap')[['count']].rename(columns={'count': 'gap'})

combined = gap_df.join(post_gap_df, how='outer').fillna(0).astype(int)
combined['total'] = combined.sum(axis=1)
combined['pct_no_gap'] = (combined['no_gap'] / combined['total'] * 100).round(1)
combined['pct_gap'] = (combined['gap'] / combined['total'] * 100).round(1)

print("=== POS BY GAP vs NO_GAP (Wilcox 2022) ===")
print(combined[['no_gap', 'pct_no_gap', 'gap', 'pct_gap', 'total']])

=== POS BY GAP vs NO_GAP (Wilcox 2022) ===
       no_gap  pct_no_gap   gap  pct_gap  total
pos                                            
ADJ        10         2.5   390     97.5    400
ADP         0         0.0  1370    100.0   1370
ADV         0         0.0    64    100.0     64
DET       922        99.6     4      0.4    926
NOUN       14        11.9   104     88.1    118
NUM         0         0.0    20    100.0     20
PART        0         0.0     4    100.0      4
PRON      322       100.0     0      0.0    322
SCONJ       0         0.0    22    100.0     22
VERB        4         1.9   206     98.1    210
